In [ ]:
!pip install -q langchain langgraph langchain-huggingface langchain-chroma sentence-transformers transformers accelerate
!pip install langchain_community pypdf langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [ ]:
from typing import TypedDict, List, Annotated
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
import operator
import urllib.request
import torch
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

/tmp/ipykernel_2073/907837285.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# Self-RAG 함수 정의

`SelfRAGState`: 노드끼리 주고받는 데이터 상자

질문, 검색 문서, 답변, 평가 결과, 평가 횟수
- retry_count: 평가할 때마다 +1

In [ ]:
class SelfRAGState(TypedDict):
    query: str
    retrieved_docs: List[Document]
    answer: str
    evaluation: str        # "sufficient" 또는 "insufficient"
    retry_count: Annotated[int, operator.add]

`retrieve`: 검색

질문으로 벡터 DB에서 관련 문서 3개를 찾아 retrieved_docs에 저장

In [ ]:
def retrieve(state: SelfRAGState) -> dict:
    docs = vector_store.similarity_search(state["query"], k=3)
    return {"retrieved_docs": docs}

`generate`: 답변 생성


- 검색된 문서를 이어 붙여 컨텍스트로 넣고

- LLM이 답변을 만들어 answer에 저장함.

- 문서에 정보가 부족하면 그렇다고 밝히라고 지시

In [ ]:
def generate(state: SelfRAGState) -> dict:
    context = "\n\n".join(d.page_content for d in state["retrieved_docs"])
    prompt = ChatPromptTemplate.from_template(
        "컨텍스트:\n{context}\n\n질문: {query}\n\n"
        "컨텍스트를 바탕으로 답변하세요. "
        "컨텍스트에 정보가 부족하면 그 사실을 명시하세요."
    )
    response = llm.invoke(prompt.format(context=context, query=state["query"]))
    return {"answer": response.content}

`evaluate`: 답변 평가

질문, 문서, 답변을 LLM에게 보여주고 "문서에 충실하고 질문에 맞는가?"를

sufficient / insufficient로 판단
- 평가할 때마다 retry_count +1

In [ ]:
def evaluate(state: SelfRAGState) -> dict:
    """생성된 답변의 충실도와 관련성을 평가합니다."""
    context = "\n\n".join(d.page_content for d in state["retrieved_docs"])
    prompt = ChatPromptTemplate.from_template(
        "질문: {query}\n"
        "컨텍스트: {context}\n"
        "답변: {answer}\n\n"
        "답변이 컨텍스트에 충실하고 질문에 적절히 답하고 있는지 평가하세요.\n"
        "'sufficient' 또는 'insufficient'로만 답하세요."
    )
    response = llm.invoke(prompt.format(
        query=state["query"],
        context=context,
        answer=state["answer"],
    ))
    evaluation = "insufficient" if "insufficient" in response.content.lower() else "sufficient"
    return {"evaluation": evaluation, "retry_count": 1}

`refine`: 답변 수정


같은 문서를 보고 이전 답변을 더 정확하게 고쳐 써서 answer를 새 답변으로 바꿈
- 재검색 X

In [ ]:
def refine(state: SelfRAGState) -> dict:
    """평가 결과를 반영하여 답변을 개선합니다."""
    context = "\n\n".join(d.page_content for d in state["retrieved_docs"])
    prompt = ChatPromptTemplate.from_template(
        "다음 답변을 개선하세요.\n\n"
        "원래 질문: {query}\n"
        "컨텍스트: {context}\n"
        "이전 답변: {answer}\n\n"
        "컨텍스트에 더 충실하고, 질문에 더 정확히 답하도록 수정하세요."
    )
    response = llm.invoke(prompt.format(
        query=state["query"],
        context=context,
        answer=state["answer"],
    ))
    return {"answer": response.content}

`route_after_eval`: 다음 경로 결정

평가가 sufficient면 → 종료

평가를 이미 2번 했으면 → 종료 (무한 반복 방지)

그 외 → 수정(refine)으로

In [ ]:
def route_after_eval(state: SelfRAGState) -> str:
    if state["evaluation"] == "sufficient":
        return "done"
    if state.get("retry_count", 0) >= 2:
        return "done"
    return "refine"

`그래프 구성` : 노드 등록과 연결

In [ ]:
graph = StateGraph(SelfRAGState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_node("evaluate", evaluate)
graph.add_node("refine", refine)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges(
    "evaluate",
    route_after_eval,
    {"done": END, "refine": "refine"},
)
graph.add_edge("refine", "evaluate")

실행 가능한 앱으로 완성

In [ ]:
self_rag_app = graph.compile()

# RAG 파이프라인

데이터 다운로드

In [ ]:
# 분석할 PDF 파일을 웹에서 다운로드.
urllib.request.urlretrieve("https://github.com/llama-index-tutorial/llama-index-tutorial/raw/main/ch07/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf", filename="2023_북한인권보고서.pdf")

('2023_북한인권보고서.pdf', <http.client.HTTPMessage at 0x7e391b97c180>)

LLM 설정

In [ ]:
llm = ChatHuggingFace(llm=HuggingFacePipeline.from_model_id(
    model_id='Qwen/Qwen2.5-3B-Instruct',        # GPT-4o 대신 사용할 LLM
    task='text-generation',
    device=0,
    model_kwargs={'torch_dtype':torch.float16},
    pipeline_kwargs={
        'max_new_tokens': 500,
        'do_sample': False,
        'return_full_text': False,
    },
))

임베딩 모델

In [ ]:
embed_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',                    # text-embedding-3-large 대신 사용할 임베딩
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True},
)

문서 분할 설정

In [ ]:
# 문서 분할 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,  # 문서를 300자 단위로 분할
    chunk_overlap=100,  # 문맥 유지를 위해 청크 간 100자 중복
)

pdf load & embedding

In [ ]:
loader = PyPDFLoader("2023_북한인권보고서.pdf")  # PDF 문서 로더
documents = loader.load()  # 문서에서 텍스트 추출
chunks = text_splitter.split_documents(documents)  # 문서 분할

In [ ]:
# vector_store.delete_collection()
vector_store = Chroma.from_documents(chunks, embed_model)
print("벡터스토어 문서 수:", vector_store._collection.count())   # 청크 수와 같아야 함

벡터스토어 문서 수: 1835


### Self-RAG 실행

In [ ]:
question = "북한 사람들은 어떤 단체에 의무적으로 들어가야 해?"

for step in self_rag_app.stream({"query": question, "retry_count": 0}):
    for node, output in step.items():
        print(f"\n===== [{node}] =====")
        if "retrieved_docs" in output:
            for d in output["retrieved_docs"]:
                print(f"- p.{d.metadata.get('page')} {' '.join(d.page_content.split())[:60]}...")
        if "answer" in output:
            print(output["answer"])
        if "evaluation" in output:
            print(f"평가: {output['evaluation']}")


===== [retrieve] =====
- p.31 당국의 행사와 각종 궐기대회에 집단적으로 동원되고 있으며, 동원 에 불참하면 당에 대한 충성심에 문제가 있다...
- p.208 년단(이하 소년단), 사회주의애국청년동맹(이하 청년동맹), 조선직 업총동맹(이하 직맹), 조선농업근로자동맹(...
- p.207 2023 북한인권보고서 206 체가 동원되거나, 특정 회사가 지정되 면 그 조직 에 소속된 사람들 이 동원되...


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the docum


===== [generate] =====
북한 사람들은 노동당의 지도 및 감독을 받는 다양한 사회단체에 의무적으로 가입해야 합니다. 이들 단체들은 다음과 같습니다:

1. 소년단 (소학교 2학년부터)
2. 사회주의애국청년동맹 (청년동맹)
3. 조선직업총동맹 (직맹)
4. 조선농업근로자동맹 (농근맹)
5. 조선사회주의여성동맹 (여맹)

이러한 단체들은 모두 노동당의 지도 및 감독을 받아야 하며, 북한 주민들은 이러한 가입을 거부할 수 없으며, 그렇지 않으면 비판을 받을 위험이 있습니다. 이는 북한 주민들이 조직생활에 강제적으로 참여하도록 하는 것으로, 이는 결사의 자유를 침해하고 있습니다.

===== [evaluate] =====
평가: sufficient


In [ ]:
question = "북한의 2023년 경제성장률은 얼마야?"

for step in self_rag_app.stream({"query": question, "retry_count": 0}):
    for node, output in step.items():
        print(f"\n===== [{node}] =====")
        if "retrieved_docs" in output:
            for d in output["retrieved_docs"]:
                print(f"- p.{d.metadata.get('page')} {' '.join(d.page_content.split())[:60]}...")
        if "answer" in output:
            print(output["answer"])
        if "evaluation" in output:
            print(f"평가: {output['evaluation']}")

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



===== [retrieve] =====
- p.19 2023 북한인권보고서...
- p.53 2023 북한인권보고서...
- p.415 2023 북한인권보고서...


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



===== [generate] =====
주어진 컨텍스트에는 2023년 북한의 경제성장률에 대한 정보는 포함되어 있지 않습니다. 이에 대한 정확한 데이터를 제공하기 위해서는 해당 논문이나 보고서를 직접 확인해야 합니다. 다른 정보나 추정치는 제공할 수 없으니, 원활한 대화를 위해 필요한 정보는 가능한 한 자세히 알려주시면 감사하겠습니다.

===== [evaluate] =====
평가: insufficient


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



===== [refine] =====
다음과 같이 개선하겠습니다:

2023 북한인권보고서에 따르면, 2023년 북한의 경제성장률에 대한 정확한 정보는 제공되지 않았습니다. 이에 대한 정확한 데이터를 얻기 위해서는 해당 보고서를 다시 확인하거나 다른 공신력 있는 자료를 참조해야 할 것입니다. 

추가로, 보고서에서는 북한의 경제 상황에 대한 일반적인 평가와 문제점들을 다루고 있으나, 특정 연도별 성장률에 대한 세부 정보는 제공되지 않았음을 알려드립니다.

===== [evaluate] =====
평가: insufficient
